In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

In [5]:
# 1. Load dataset
df = pd.read_csv("Iris.csv")
X = df.drop(columns=["Id", "Species"], errors="ignore")
y = LabelEncoder().fit_transform(df["Species"])

In [7]:
# 2. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
# 3. Create boosting models
ada = AdaBoostClassifier(n_estimators=50, random_state=42)
gbm = GradientBoostingClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    objective="multi:softprob", num_class=3,
    eval_metric="mlogloss", random_state=42
)


In [11]:
# 4. Create voting classifiers
hard = VotingClassifier(
    estimators=[("ada", ada), ("gbm", gbm), ("xgb", xgb)],
    voting="hard"
)
soft = VotingClassifier(
    estimators=[("ada", ada), ("gbm", gbm), ("xgb", xgb)],
    voting="soft"
)

models = {
    "AdaBoost": ada,
    "Gradient Boosting": gbm,
    "XGBoost": xgb,
    "Hard Voting": hard,
    "Soft Voting": soft
}


In [13]:
# 5. Train models and compare performance
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Error Rate": 1 - accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test, pred, average="weighted", zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, average="weighted", zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, pred, average="weighted", zero_division=0
        )
    })

In [15]:
# 6. Display comparison table
print(pd.DataFrame(results).round(3).to_string(index=False))

            Model  Accuracy  Error Rate  Precision  Recall  F1 Score
         AdaBoost     0.933       0.067      0.933   0.933     0.933
Gradient Boosting     0.967       0.033      0.970   0.967     0.967
          XGBoost     0.933       0.067      0.933   0.933     0.933
      Hard Voting     0.967       0.033      0.970   0.967     0.967
      Soft Voting     0.967       0.033      0.970   0.967     0.967
